In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/TTBK1')

import pandas as pd

# ──────────────────────────────────────────────────────────────────────────────
# Utility Functions
# ──────────────────────────────────────────────────────────────────────────────
def generate_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol) if mol else None

def smiles_to_fp(smiles_list, radius=2, nbits=2048):
    fps = []
    for smi in smiles_list:
        m = Chem.MolFromSmiles(smi)
        if m:
            bit = AllChem.GetMorganFingerprintAsBitVect(m, radius, nbits)
            arr = np.zeros((nbits,), dtype=int)
            DataStructs.ConvertToNumpyArray(bit, arr)
            fps.append(arr)
    return np.array(fps)

def load_and_process(path):
    df = pd.read_csv(path).dropna(subset=['smiles', 'label'])
    df['label'] = df['label'].astype(int)
    df = df[df['smiles'].apply(lambda s: Chem.MolFromSmiles(s) is not None)]
    df['scaffold'] = df['smiles'].apply(generate_scaffold)
    return df

# ──────────────────────────────────────────────────────────────────────────────
# Nested CV Evaluation + Save Best Estimators
# ──────────────────────────────────────────────────────────────────────────────
def nested_cv_evaluation_and_save(X, y, models, param_grids, save_prefix="model"):
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    for model_name, model in models.items():
        print(f"\n[⏳] Running Nested CV for {model_name}...")
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        grid = GridSearchCV(model, param_grids[model_name], cv=inner_cv,
                            scoring='f1_macro', n_jobs=1)

        f1_scores, roc_scores, pr_scores = [], [], []
        smote_increases = []  # [ADDED] To track SMOTE increases per fold

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            orig_train_count = len(y_train)
            k_neighbors = min(5, np.bincount(y_train).min() - 1)
            sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
            X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
            resampled_train_count = len(y_train_res)
            increase = resampled_train_count - orig_train_count
            smote_increases.append(increase)
            print(f"[SMOTE] Fold {fold}: Train size before={orig_train_count}, after={resampled_train_count}, increased by {increase}")

            grid.fit(X_train_res, y_train_res)
            best_model = grid.best_estimator_

            y_pred = best_model.predict(X_test)
            y_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else None

            f1_scores.append(f1_score(y_test, y_pred))
            roc_scores.append(roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan)
            pr_scores.append(average_precision_score(y_test, y_proba) if y_proba is not None else np.nan)

        # [FIXED] Calculate total_increase here, before using it
        print(f"[] Refitting best {model_name} model on full dataset for saving...")
        k_neighbors = min(5, np.bincount(y).min() - 1)
        sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
        orig_total_count = len(y)
        X_bal, y_bal = sm.fit_resample(X, y)
        resampled_total_count = len(y_bal)
        total_increase = resampled_total_count - orig_total_count
        print(f"[SMOTE] Full data: size before={orig_total_count}, after={resampled_total_count}, increased by {total_increase}")

        grid.fit(X_bal, y_bal)
        final_model = grid.best_estimator_

        # fname = f"{save_prefix}_{model_name.lower()}.pkl"
        # joblib.dump(final_model, fname)
        # print(f"[💾] Saved {model_name} model to: {fname}")

        results.append({
            "Model": model_name,
            "F1_mean": np.mean(f1_scores),
            "F1_std": np.std(f1_scores),
            "ROC_AUC_mean": np.mean(roc_scores),
            "PR_AUC_mean": np.mean(pr_scores),
            "Avg_SMOTE_increase_per_fold": np.mean(smote_increases),  # [ADDED]
            "Total_SMOTE_increase_on_full": total_increase           # [ADDED]
        })

    return pd.DataFrame(results)




# ──────────────────────────────────────────────────────────────────────────────
# Main Script
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = load_and_process("dedeuplicated_combined_ic50.csv")
    X = smiles_to_fp(df['smiles'].values)
    y = df['label'].values
    # do a one-time split and save it
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        stratify=y,
        random_state=42
    )
    # save them to disk
    np.savez_compressed(
        "train_test_data.npz",
        X_train=X_train,
        X_test =X_test,
        y_train=y_train,
        y_test =y_test
    )
    print("✅ Saved fixed train/test split to train_test_data.npz")


    models = {
        "CatBoost": CatBoostClassifier(task_type="GPU", devices="0", verbose=0, random_state=42),
        "SVM": Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
        "KNN": Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())]),
        "NaiveBayes": GaussianNB(),
        "XGB": XGBClassifier(tree_method="gpu_hist", predictor="gpu_predictor",
                             use_label_encoder=False, eval_metric='logloss', random_state=42)
    }

    param_grids = {
        "CatBoost": {'depth': [4, 6], 'learning_rate': [0.01], 'iterations': [100]},
        "SVM": {'svc__C': [0.1, 1], 'svc__gamma': ['scale']},
        "KNN": {'knn__n_neighbors': [3, 5]},
        "NaiveBayes": {'var_smoothing': [1e-9, 1e-8]},
        "XGB": {'max_depth': [3, 5], 'learning_rate': [0.01], 'n_estimators': [100]}
    }

    results_df = nested_cv_evaluation_and_save(X, y, models, param_grids)
    print("\n[NESTED CV RESULTS]")
    print(results_df.to_string(index=False))



[⏳] Running Nested CV for CatBoost...
[SMOTE] Fold 1: Train size before=99, after=120, increased by 21
[SMOTE] Fold 2: Train size before=99, after=120, increased by 21
[SMOTE] Fold 3: Train size before=99, after=120, increased by 21
[SMOTE] Fold 4: Train size before=99, after=120, increased by 21
[SMOTE] Fold 5: Train size before=100, after=120, increased by 20
[] Refitting best CatBoost model on full dataset for saving...
[SMOTE] Full data: size before=124, after=150, increased by 26

[⏳] Running Nested CV for SVM...
[SMOTE] Fold 1: Train size before=99, after=120, increased by 21
[SMOTE] Fold 2: Train size before=99, after=120, increased by 21
[SMOTE] Fold 3: Train size before=99, after=120, increased by 21
[SMOTE] Fold 4: Train size before=99, after=120, increased by 21
[SMOTE] Fold 5: Train size before=100, after=120, increased by 20
[] Refitting best SVM model on full dataset for saving...
[SMOTE] Full data: size before=124, after=150, increased by 26

[⏳] Running Nested CV for K

In [ ]:
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT')
import pandas as pd

# Load all three CSV files
df1 = pd.read_csv('Affinity_result_file_ABL1_1000.csv')
df2 = pd.read_csv('Affinity_result_file_DYRK1A_1000.csv')
df3 = pd.read_csv('Affinity_result_file_TTBK1_1000.csv')

# Extract the 'name of the ligand' column as sets
set1 = set(df1['Name of the ligand'])
set2 = set(df2['Name of the ligand'])
set3 = set(df3['Name of the ligand'])


# Find common ligands
common_ligands = set(df1['Name of the ligand']) & set(df2['Name of the ligand']) & set(df3['Name of the ligand'])
print(f"Number of common ligands: {len(common_ligands)}")
print(f"Number of common ligands: {len(common_ligands)}")


# Filter each dataframe for only common ligands
df1_common = df1[df1['Name of the ligand'].isin(common_ligands)]
df2_common = df2[df2['Name of the ligand'].isin(common_ligands)]
df3_common = df3[df3['Name of the ligand'].isin(common_ligands)]

# Merge all into a single dataframe on 'name of the ligand'
merged_df = df1_common.merge(df2_common, on='Name of the ligand', suffixes=('_p1', '_p2'))
merged_df = merged_df.merge(df3_common, on='Name of the ligand')

# Optional: Add suffixes to df3 columns to avoid clashes
merged_df.columns = ['Name of the ligand'] + [c if c=='Name of the ligand' else c+'_p3' for c in merged_df.columns[1:]]

# Save merged file
merged_df.to_csv('common_compounds_all_proteins.csv', index=False)
print("✅ Saved merged data for common compounds in common_compounds_all_proteins.csv")



In [6]:
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/gnina')
import pandas as pd

# Load your filtered data
final_df = pd.read_csv('combined_cnn_mean.csv')

# Load the coconut data
coconut_df = pd.read_csv('coconut_csv-06-2025.csv')

# Preview to identify the correct identifier column names
print(final_df.columns)
print(coconut_df.columns)

# Assume the identifier in coconut_df is 'coconut_ID' (replace with actual column if different)
# Merge on the identifier, keeping all rows from final_df
merged_df = final_df.merge(
    coconut_df[['Identifier', 'canonical_smiles', 'name','iupac_name']], 
    left_on='IDENTIFIER', 
    right_on='Identifier', 
    how='left'
)


# Save the merged file
merged_df.to_csv('combined_annotated.csv', index=False)


/tmp/ipykernel_5240/2258479473.py:9: DtypeWarning: Columns (6,7,8,9,10,12,13,14,15,16,17,18,19,20,21,23,24,25,26,27,28,30,38,39,41,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,66,68) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut_df = pd.read_csv('coconut_csv-06-2025.csv')


Index(['IDENTIFIER', 'CNNaffinity_abl1', 'CNNaffinity_dyrk1a',
       'CNNaffinity_ttbk1', 'CNN_mean'],
      dtype='object')
Index(['Identifier', 'canonical_smiles', 'standard_inchi',
       'standard_inchi_key', 'name', 'iupac_name', 'annotation_level',
       'total_atom_count', 'heavy_atom_count', 'molecular_weight',
       'exact_molecular_weight', 'molecular_formula', 'alogp',
       'topological_polar_surface_area', 'rotatable_bond_count',
       'hydrogen_bond_acceptors', 'hydrogen_bond_donors',
       'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski',
       'lipinski_rule_of_five_violations', 'aromatic_rings_count',
       'qed_drug_likeliness', 'formal_charge', 'fractioncsp3',
       'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar',
       'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework',
       'np_likeness', 'chemical_class', 'chemical_sub_class',
       'chemical_super_class', 'direct_parent_classification',
       

In [3]:
# final_df["canonical smiles"].to_csv("multi_target_YoudenJ_only smiles.csv", index=False)
merged_df[["canonical_smiles", "Identifier"]].to_csv("multi_target_hits_final_33_remaining.smi", sep=" ", index=False, header=False)



In [ ]:
#replacing identifier _ to .
import pandas as pd

# 1) Load your common file
df = pd.read_csv('final_common_with_smiles_and_name.csv')  # adjust filename as needed

# 2) Remove the trailing “_<number>” from each identifier
#    This will turn e.g. "CNP0381179_1_1" → "CNP0381179_1"
df['Identifier'] = df['Identifier'].str.replace(r'_\d+$', '', regex=True)

# 3) (Optional) If you also want to remove the middle “_1” and keep only the base CNP code:
#    df['identifier'] = df['identifier'].str.replace(r'_\d+_\d+$', '', regex=True)

# 4) Save back out
df.to_csv('final_common_filtered_trimmed.csv', index=False)

print("✅ identifiers have been trimmed and saved to final_common_filtered_trimmed.csv")


✅ identifiers have been trimmed and saved to final_common_filtered_trimmed.csv
